# Validação Agrupada (`GroupKFold`) e Bloqueios Anti-Vazamento

Neste notebook mostramos:
1. Como isolar pacientes/sujeitos entre **desenvolvimento e holdout** e entre **treino e validação**;
2. **Demonstrações ao vivo ('O que acontece se...')** do framework bloqueando erros metodológicos comuns.

In [ ]:
from experimental_ml.core.enums import TaskType
from experimental_ml.core.findings import ExperimentalMLError
from experimental_ml.data import (
    DatasetSchemaSpec,
    create_preparation_plan,
    load_dataset,
    profile_dataset,
    sanitize_dataset,
)
from experimental_ml.models.preprocessing import (
    PreprocessingStepSpec,
    ScalerType,
    TwoLayerPreprocessingSpec,
    validate_two_layer_preprocessing,
)
from experimental_ml.validation import (
    EvaluationRegime,
    PartitionConfig,
    PartitionStrategy,
    build_partition_plan,
    validate_partition_plan,
)

## 1. Particionamento `GroupKFold` com Holdout Isolado (`CV_AND_HOLDOUT`)

In [ ]:
dados = {
    'paciente_id': [f'P{g}' for g in range(1, 7) for _ in range(3)],
    'biomarcador_a': [float(i) * 0.1 for i in range(18)],
    'biomarcador_b': [float(18 - i) * 0.2 for i in range(18)],
    'diagnostico': [0 if g <= 3 else 1 for g in range(1, 7) for _ in range(3)],
}
schema = DatasetSchemaSpec(
    target_col='diagnostico',
    task_type=TaskType.BINARY_CLASSIFICATION,
    group_col='paciente_id',
)
bundle = load_dataset(dados, schema)
profile = profile_dataset(bundle)
view = sanitize_dataset(bundle, profile, create_preparation_plan(profile, rules=[]))

part_plan = build_partition_plan(
    view,
    PartitionConfig(
        strategy=PartitionStrategy.GROUP_KFOLD,
        regime=EvaluationRegime.CV_AND_HOLDOUT,
        n_splits=3,
        holdout_fraction=0.33,
        group_col='paciente_id',
        master_seed=42,
    ),
)
validate_partition_plan(part_plan, view)
assert set(part_plan.development_ids).isdisjoint(set(part_plan.holdout_ids))
print(f'Desenvolvimento: {len(part_plan.development_ids)} obs | Holdout: {len(part_plan.holdout_ids)} obs')

## 2. Laboratório de Bloqueios ('O que acontece se...')

Verificamos na prática que o framework recusa configurações metodologicamente inválidas.

In [ ]:
# Bloqueio 1: GROUP_KFOLD sem informar group_col
try:
    PartitionConfig(strategy=PartitionStrategy.GROUP_KFOLD, regime=EvaluationRegime.CV_ONLY, n_splits=3)
    raise AssertionError('Deveria ter bloqueado GROUP_KFOLD sem group_col!')
except ExperimentalMLError as exc:
    print('[BLOQUEIO ESPERADO 1]:', exc)

# Bloqueio 2: Escalonamento idêntico duplicado nas 2 camadas (Cenário + Modelo)
try:
    validate_two_layer_preprocessing(
        TwoLayerPreprocessingSpec(
            layer1_steps=(PreprocessingStepSpec(step_id='s1', transformer_type=ScalerType.STANDARD.value),),
            layer2_steps=(PreprocessingStepSpec(step_id='s2', transformer_type=ScalerType.STANDARD.value),),
        )
    )
    raise AssertionError('Deveria ter bloqueado escalonamento duplicado!')
except ExperimentalMLError as exc:
    print('[BLOQUEIO ESPERADO 2]:', exc)